Một chiến lược mua: Khi giá dự đoán tăng so với giá hiện tại, Momentum dương (điều này cho thấy xu hướng giá đang tăng), và RSI dưới 30 (điều này cho thấy rằng cổ phiếu không bị mua quá mức).

Một chiến lược bán: Khi giá dự đoán giảm so với giá hiện tại, Momentum âm (điều này cho thấy xu hướng giá đang giảm), và RSI trên 70 (điều này cho thấy cổ phiếu không bị bán quá mức).

# Load du lieu len

In [1]:
import sys
sys.path.append('../Common')

from CommonBinance import CommonBinance
# import CommonBinance

from datetime import datetime, timedelta

symbol = 'ETHUSDT'
from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
timeframe = '1m' # 1m

data = CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, timeframe)
# data = CommonBinance.CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, timeframe)

data

,Datetime,Open,High,Low,Close,Volume
0,2025-10-22 00:00:00,3873.04,3873.04,3862.16,3864.99,1188.6477
1,2025-10-22 00:01:00,3864.98,3868.76,3862.67,3868.76,332.1784
2,2025-10-22 00:02:00,3868.75,3872.91,3868.65,3872.91,340.9633
3,2025-10-22 00:03:00,3872.91,3877.00,3872.90,3877.00,209.3692
4,2025-10-22 00:04:00,3877.00,3878.70,3874.81,3877.14,339.9579
...,...,...,...,...,...,...
802,2025-10-22 13:22:00,3853.49,3853.95,3852.43,3853.64,131.7644
803,2025-10-22 13:23:00,3853.64,3855.83,3852.45,3852.90,114.1686
804,2025-10-22 13:24:00,3852.90,3854.99,3851.40,3853.39,110.8989
805,2025-10-22 13:25:00,3853.39,3854.44,3853.01,3853.75,60.3489


# Tim tham so p, d, q toi uu: Tim 1 lan thoi

In [ ]:
# import itertools
# import statsmodels.api as sm
# import pandas as pd

# # Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
# # data.set_index('Datetime', inplace=True) da dat o tren

# # Xác định khoảng giá trị cho tham số p, d, q
# p = d = q = range(0, 6) # 0 -> 5

# pdq = list(itertools.product(p, d, q))

# best_aic = float("inf")
# best_pdq = None
# best_model = None

# for param in pdq:
#     try:
#         model = sm.tsa.ARIMA(data['Close'], order=param)
#         results = model.fit()
#         if results.aic < best_aic:
#             best_aic = results.aic
#             best_pdq = param
#             best_model = results
#     except:
#         continue

# print(f'Best ARIMA{best_pdq} AIC:{best_aic}')

# Tinh toan chien luoc

In [2]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
import sys

# Giả sử `data` là DataFrame chứa các cột Datetime, Open, High, Low, Close, Volume
# Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
data.set_index('Datetime', inplace=True)

# Tính Momentum
data['Momentum'] = data['Close'].diff()

# Tính RSI
delta = data['Close'].diff()
up, down = delta.clip(lower=0), delta.clip(upper=0).abs()
roll_up, roll_down = up.rolling(14).mean(), down.rolling(14).mean()
RS = roll_up / roll_down
data['RSI'] = 100.0 - (100.0 / (1.0 + RS))

# Chọn tham số ARIMA dựa trên AIC, BIC hoặc các tiêu chí khác (cần phải thực hiện trước) => Chay cell o tren
model = ARIMA(data['Close'], order=(5, 1, 5))
model_fit = model.fit()

# Dự đoán giá
data['Predicted_Close'] = model_fit.predict(start=0, end=len(data)-1, typ='levels')

# Xác định tín hiệu mua và bán
data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['Momentum'] > 0) & (data['RSI'] < 30))
data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 10) & (data['RSI'] > 50))

# Trực quan hóa
import plotly.graph_objects as go

# Tạo biểu đồ cơ bản với giá đóng cửa
fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close'))

# Thêm dữ liệu giá dự đoán vào biểu đồ
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Predicted Close', line=dict(dash='dot', color='red')))

# Thêm điểm mua
buy_signals = data[data['Buy_Signal']]
fig.add_trace(go.Scatter(x=buy_signals.index, y=buy_signals['Close'], mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')))

# Thêm điểm bán
sell_signals = data[data['Sell_Signal']]
fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['Close'], mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')))

# Tùy chỉnh layout
fig.update_layout(title='Trading Signals with ARIMA, Momentum, and RSI', xaxis_title='Date', yaxis_title='Price', hovermode='x')

# Hiển thị biểu đồ
fig.show()

c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [11]:
data

,Open,High,Low,Close,Volume,Momentum,RSI,Predicted_Close,Buy_Signal,Sell_Signal
Datetime,,,,,,,,,,
2025-03-20 00:00:00,2056.05,2058.41,2056.05,2056.29,472.1212,NaN,NaN,0.000000,False,False
2025-03-20 00:01:00,2056.29,2056.47,2054.39,2055.19,322.5878,-1.10,NaN,2056.289880,False,False
2025-03-20 00:02:00,2055.19,2059.59,2054.90,2057.61,826.1715,2.42,NaN,2055.228319,False,False
2025-03-20 00:03:00,2057.60,2060.48,2056.52,2060.48,426.5250,2.87,NaN,2057.468921,False,False
2025-03-20 00:04:00,2060.48,2066.96,2060.48,2061.59,1311.6063,1.11,NaN,2060.453958,False,False
...,...,...,...,...,...,...,...,...,...,...
2025-03-20 13:58:00,1983.23,1984.80,1983.22,1984.52,134.1751,1.30,33.694839,1982.885954,False,False
2025-03-20 13:59:00,1984.51,1986.30,1984.05,1985.98,243.9435,1.46,28.392569,1984.339189,False,False
2025-03-20 14:00:00,1985.98,1991.04,1984.84,1990.49,638.2149,4.51,40.748588,1986.302382,False,False


In [ ]:
import plotly.graph_objects as go

# Giả sử DataFrame của bạn tên là `data` và có chỉ mục datetime
fig = go.Figure()

# Thêm đường biểu diễn giá đóng cửa và giá dự đoán
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close'))
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Predicted Close'))

# Thêm điểm cho tín hiệu mua và bán
fig.add_trace(go.Scatter(x=data[data['Buy_Signal']].index, y=data[data['Buy_Signal']]['Close'], 
                         mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')))
fig.add_trace(go.Scatter(x=data[data['Sell_Signal']].index, y=data[data['Sell_Signal']]['Close'], 
                         mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')))

# Thêm đường cho Momentum và RSI với trục y phụ
fig.add_trace(go.Scatter(x=data.index, y=data['Momentum'], name='Momentum', yaxis='y2'))
fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], name='RSI', yaxis='y3'))

# Tùy chỉnh layout cho các trục y phụ
fig.update_layout(
    title='Trading Signals with ARIMA, Momentum, and RSI',
    xaxis_title='Date',
    yaxis_title='Close Price',
    yaxis2=dict(
        title='Momentum',
        titlefont=dict(color='purple'),
        tickfont=dict(color='purple'),
        overlaying='y',
        side='right'
    ),
    yaxis3=dict(
        title='RSI',
        titlefont=dict(color='orange'),
        tickfont=dict(color='orange'),
        overlaying='y',
        side='right',
        position=0.95
    )
)

# Hiển thị biểu đồ
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Mã chuẩn bị dữ liệu ở đây
# Giả sử 'data' là DataFrame của bạn và nó có 'Datetime' làm chỉ mục

# Tạo biểu đồ con: 3 hàng cho Giá đóng cửa, Momentum, và RSI
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Trading Signals with ARIMA, Momentum, and RSI', 'Momentum', 'RSI'),
                    vertical_spacing=0.1)

# Thêm giá đóng và dự đoán giá đóng vào hàng đầu tiên
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Giá Đóng'), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Dự Đoán Giá Đóng'), row=1, col=1)

# Thêm tín hiệu mua và bán trên cùng một biểu đồ con
fig.add_trace(go.Scatter(x=data[data['Buy_Signal']].index, y=data[data['Buy_Signal']]['Close'], mode='markers',
                         marker=dict(color='green', size=10, symbol='triangle-up'), name='Tín Hiệu Mua'), row=1, col=1)
fig.add_trace(go.Scatter(x=data[data['Sell_Signal']].index, y=data[data['Sell_Signal']]['Close'], mode='markers',
                         marker=dict(color='red', size=10, symbol='triangle-down'), name='Tín Hiệu Bán'), row=1, col=1)

# Thêm Momentum vào hàng thứ hai
fig.add_trace(go.Scatter(x=data.index, y=data['Momentum'], mode='lines', name='Momentum'), row=2, col=1)

# Thêm RSI vào hàng thứ ba
fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], mode='lines', name='RSI'), row=3, col=1)

# Cập nhật layout
fig.update_layout(height=800, title_text="Chiến Lược Giao Dịch với Tín Hiệu Mua/Bán, Momentum và RSI")

# Hiển thị biểu đồ
fig.show()
